# Setup
Notebooks call reusable functions from `src/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve_paths, set_global_seed, get_seed
config = load_config(ROOT / 'configs/project_config.yaml')
paths = resolve_paths(config)
set_global_seed(get_seed(config))
print('project root:', paths.root)


## Data audit and EDA

In [ ]:
from src.data_loader import summarize_raw_catalog, load_btcirt
from src.data_validation import audit_order_book_quality, audit_timestamp_gaps, validate_schema, save_quality_report
catalog = summarize_raw_catalog(paths.raw)
print(catalog)
df = load_btcirt(paths.raw, config)
schema = validate_schema(df)
book = audit_order_book_quality(df)
gaps = audit_timestamp_gaps(df.sort_values('timestamp'))
save_quality_report(catalog, schema, book, gaps, paths.metrics, paths.tables)
print(gaps.get('grid_alignment_decision'))


In [ ]:
from src.visualization import plot_timestamp_gap_hist, plot_snapshots_by_date, plot_mid_price
import pandas as pd
ts = df.sort_values('timestamp')
gap = ts['timestamp'].diff().dt.total_seconds().dropna()
plot_timestamp_gap_hist(gap, paths.figures/'timestamp_gap_hist.png')
plot_snapshots_by_date(ts.assign(timestamp=pd.to_datetime(ts['timestamp'])), paths.figures/'snapshots_by_date.png')
